# Демо 1. Класичне донавчання: зміна голови в моделі BERT

У цьому ноутбуці донавчаємо модель **BERT** для задачі **визначення парафраз**: модель отримує два речення й визначає, чи передають вони той самий зміст.

План:
1. Завантажити датасет MRPC і токенізувати пари речень.
2. Завантажити BERT із **новою головою**-класифікатором.
3. **Повне донавчання** з `Trainer`: тренуються всі ваги (кістяк і голова).
4. Перевірити модель на власних прикладах.

Приклад за мотивами розділу 3 курсу [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter3/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [1]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [2]:
# Фіксуємо seed, щоб результати повторювалися між запусками
set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Пристрій: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
else:
    print("Включіть GPU, бо набридне чекати ;)")

Пристрій: cuda
GPU: Tesla T4


## 1. Дані: MRPC

**MRPC** (Microsoft Research Paraphrase Corpus) — пари речень з новинних статей.
Мітка `1` означає, що речення є парафразами, мітка `0` — що ні.

In [3]:
raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

mrpc/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  649kB            

mrpc/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 75.7kB            

mrpc/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  308kB            

mrpc/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [4]:
# Подивимося на один приклад
print("Приклад нееквівалентних речень:")
example = raw_datasets["train"][3]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

print("")
print("Приклад еквівалентних речень:")
example = raw_datasets["train"][5]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

Приклад нееквівалентних речень:
Речення 1: Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .
Речення 2: Tab shares jumped 20 cents , or 4.6 % , to set a record closing high at A $ 4.57 .
Мітка:    0 → not_equivalent

Приклад еквівалентних речень:
Речення 1: Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .
Речення 2: With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .
Мітка:    1 → equivalent


### Знайомимось з BERT BASE UNCASED

Для цього прочитайте інформацію на [картці моделі](https://huggingface.co/google-bert/bert-base-uncased)

In [5]:
from transformers import pipeline

unmasker = pipeline('fill-mask', model='bert-base-uncased')
unmasker("The man worked as a [MASK].")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[{'score': 0.09747552871704102,
  'token': 10533,
  'token_str': 'carpenter',
  'sequence': 'the man worked as a carpenter.'},
 {'score': 0.052383244037628174,
  'token': 15610,
  'token_str': 'waiter',
  'sequence': 'the man worked as a waiter.'},
 {'score': 0.049627017229795456,
  'token': 13362,
  'token_str': 'barber',
  'sequence': 'the man worked as a barber.'},
 {'score': 0.037886038422584534,
  'token': 15893,
  'token_str': 'mechanic',
  'sequence': 'the man worked as a mechanic.'},
 {'score': 0.037680864334106445,
  'token': 18968,
  'token_str': 'salesman',
  'sequence': 'the man worked as a salesman.'}]

In [6]:
print("Не по темі, але погляньте, яка \"несправедливість\"...")
unmasker("The woman worked as a [MASK].")

Не по темі, але погляньте, яка "несправедливість"...


[{'score': 0.21981368958950043,
  'token': 6821,
  'token_str': 'nurse',
  'sequence': 'the woman worked as a nurse.'},
 {'score': 0.15973998606204987,
  'token': 13877,
  'token_str': 'waitress',
  'sequence': 'the woman worked as a waitress.'},
 {'score': 0.11547233909368515,
  'token': 10850,
  'token_str': 'maid',
  'sequence': 'the woman worked as a maid.'},
 {'score': 0.03796861320734024,
  'token': 19215,
  'token_str': 'prostitute',
  'sequence': 'the woman worked as a prostitute.'},
 {'score': 0.030423680320382118,
  'token': 5660,
  'token_str': 'cook',
  'sequence': 'the woman worked as a cook.'}]

### Токенізація пари речень

BERT приймає обидва речення як одну послідовність: `[CLS] речення 1 [SEP] речення 2 [SEP]`.
Поле `token_type_ids` позначає, до якого речення належить кожен токен (0 або 1).

Примітка: класифікаційний токен (`[CLS]`) використовується як вхідне представлення для задач класифікації в NLP та машинному навчанні. Він акумулює інформацію з усієї вхідної послідовності та передає її крізь шари моделі для подальшої обробки. Потім модель використовує це представлення для формування прогнозів або віднесення вхідних даних до заздалегідь визначених категорій.

In [7]:
checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
encoded = tokenizer(example["sentence1"], example["sentence2"])

print("")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[:20])
print(encoded["token_type_ids"][:20])
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[-20:])
print(encoded["token_type_ids"][-20:])


['[CLS]', 'revenue', 'in', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[',', 'revenue', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.', '[SEP]']
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [8]:
def tokenize_function(batch):
    # truncation=True обрізає надто довгі пари до максимальної довжини моделі
    return tokenizer(batch["sentence1"], batch["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Доповнення (padding) робимо динамічно для кожного батча — до найдовшого прикладу в ньому
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_datasets

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

## 2. Модель: кістяк + нова голова

`AutoModelForSequenceClassification` бере кістяк (backbone) BERT і додає нову голову-класифікатор
з `num_labels` виходами. Зверніть увагу на звіт під час завантаження:
- ваги `cls.*` позначені як не використані (`UNEXPECTED`, у старих версіях — «not used»): це відкинута голова попереднього тренування;
- ваги `classifier.*` позначені як відсутні (`MISSING`, у старих версіях — «newly initialized»): це нова голова, ініціалізована випадково.

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

print(model.classifier)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Linear(in_features=768, out_features=2, bias=True)


In [10]:
def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if p.requires_grad or not trainable_only)


print(f"Кістяк (model.bert):   {count_params(model.bert):>12,} параметрів")
print(f"Голова (model.classifier): {count_params(model.classifier):>8,} параметрів")

Кістяк (model.bert):    109,482,240 параметрів
Голова (model.classifier):    1,538 параметрів


## 3. Повне донавчання з `Trainer`

`Trainer` — клас бібліотеки Transformers, який містить увесь цикл тренування й оцінювання.
Метрики рахуємо самі у функції `compute_metrics`.

Оновлюються **всі** ваги: і кістяк, і нова голова.
Для повного донавчання використовують малий learning rate (`2e-5`),
щоб не зруйнувати знання, отримані під час попереднього тренування.

In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }


training_args = TrainingArguments(
    output_dir="bert-mrpc",
    num_train_epochs=3,             # зазвичай вистачає 1-3 епохи
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",          # оцінюємо на валідації після кожної епохи
    save_strategy="no",             # чекпоінти не зберігаємо
    logging_steps=50,
    fp16=torch.cuda.is_available(), # змішана точність пришвидшує тренування на GPU
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [12]:
print(f"Параметрів, що тренуються: {count_params(model, trainable_only=True):,}")
trainer.train()
trainer.evaluate()

Параметрів, що тренуються: 109,483,778


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.487028,0.381455,0.838235,0.886986
2,0.328676,0.357605,0.848039,0.893471
3,0.167169,0.416541,0.860294,0.901554


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.167169,0.416541,3,0.860294,0.901554


{'eval_loss': 0.4165409803390503,
 'eval_accuracy': 0.8602941176470589,
 'eval_f1': 0.9015544041450777}

## 4. Перевірка на власних прикладах

In [13]:
def predict(model, sentence1, sentence2):
    model.eval()
    inputs = tokenizer(sentence1, sentence2, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = model(**inputs).logits.softmax(dim=-1)[0]
    return f"парафраз: {probs[1]:.1%}"


pairs = [
    ("The company reported a profit of $5 million.", "The firm earned five million dollars in profit."),
    ("The company reported a profit of $5 million.", "The weather was sunny all week."),
    ("She bought a new car yesterday.", "Yesterday she purchased a new vehicle."),
]
for s1, s2 in pairs:
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)}\n")

The company reported a profit of $5 million.
The firm earned five million dollars in profit.
→ парафраз: 86.0%

The company reported a profit of $5 million.
The weather was sunny all week.
→ парафраз: 2.6%

She bought a new car yesterday.
Yesterday she purchased a new vehicle.
→ парафраз: 99.0%



In [14]:
# Додаткові 4 власні перевірки (п. 4)
extra_pairs = [
    ("The city council approved the new budget on Monday.", "On Monday the city council passed the new budget."),
    ("The city council approved the new budget on Monday.", "The football team won the championship last year."),
    ("He is happy with the results of the exam.", "He is not happy with the results of the exam."),
    ("Apple released a new iPhone in September.", "A new iPhone was launched by Apple in September."),
]
for s1, s2 in extra_pairs:
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)}\n")

The city council approved the new budget on Monday.
On Monday the city council passed the new budget.
→ парафраз: 99.0%

The city council approved the new budget on Monday.
The football team won the championship last year.
→ парафраз: 3.1%

He is happy with the results of the exam.
He is not happy with the results of the exam.
→ парафраз: 99.1%

Apple released a new iPhone in September.
A new iPhone was launched by Apple in September.
→ парафраз: 98.8%



In [15]:
# Експеримент: збільшуємо кількість епох з 3 до 10 (нова модель, ті самі решта параметрів)
set_seed(42)
model_long = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

args_long = TrainingArguments(
    output_dir="bert-mrpc-long",
    num_train_epochs=10,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

trainer_long = Trainer(
    model=model_long,
    args=args_long,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer_long.train()
trainer_long.evaluate()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.471555,0.398554,0.799020,0.846442
2,0.317997,0.382299,0.845588,0.893039
3,0.169479,0.492709,0.843137,0.882353
4,0.112328,0.609630,0.865196,0.905660
5,0.030664,0.665817,0.867647,0.903571
6,0.033833,0.865115,0.855392,0.898451
7,0.028753,0.954503,0.850490,0.896785
8,0.003791,0.878268,0.855392,0.897747
9,0.000345,0.943451,0.862745,0.904110
10,0.000318,0.931049,0.860294,0.902230


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000318,0.931049,10,0.860294,0.902230


{'eval_loss': 0.9310493469238281,
 'eval_accuracy': 0.8602941176470589,
 'eval_f1': 0.902229845626072}

## Відповіді на запитання 0.1 (питання 1–10)

**1. Як довго виконувалися клітинки ноутбуку 01_bert_head? Чи були проблеми з запуском, які?**

Запуск через «Виконати всі» на GPU Tesla T4: підключення до середовища — близько 20 с, імпорти — 44 с, завантаження датасету, токенізатора й моделей — по кілька секунд, донавчання на 3 епохи — 1 хв 3 с (690 кроків). Уся послідовність разом зайняла близько 3 хвилин. Клітинка з експериментом на 10 епох (п. 9) виконувалась 3 хв 14 с. Суттєвих проблем із запуском не було. Лише перед запуском через нестабільний інтернет Colab показав помилку «Kernel: Failed to unassign endpoint»; вона зникла після відключення й повторного підключення середовища.

**2. Що таке токен [CLS]? Яким чином він виконує свою функцію?**

[CLS] (classification) — службовий токен, який BERT ставить на початок будь-якої послідовності. Він не відповідає жодному слову, тому його вихідний вектор не «прив'язаний» до конкретного слова. Завдяки механізму self-attention на кожному шарі цей токен бачить усі інші токени пари й накопичує інформацію про всю послідовність. Під час попереднього навчання (задача next sentence prediction) його вектор вчили представляти зміст усієї пари. Тому в класифікації вихід [CLS] (вектор довжини 768) подається в голову-класифікатор, яка за ним визначає клас.

**3. Яке пряме застосування базової моделі bert-base-uncased?**

Базова модель навчена на задачі маскованого мовного моделювання (masked language modeling) і next sentence prediction. Тому без донавчання її безпосередньо застосовують для заповнення пропусків [MASK] у тексті (pipeline fill-mask, як у демо) та як екстрактор ознак (embeddings) для речень і слів. Для кінцевих задач (класифікація, парафрази, NER) її зазвичай донавчають.

**4. Чи варто боротися з проявами сексизму LLM-моделей і яким чином?**

Так, варто. У демо видно упередженість: для «The man worked as a [MASK]» модель пропонує carpenter, waiter, barber, mechanic, salesman, а для «The woman worked as a [MASK]» — nurse, waitress, maid, prostitute, cook. Модель відтворює стереотипи з навчальних даних, а в реальних застосунках (відбір резюме, рекомендації, чат-боти) вони можуть призводити до дискримінації. Способи боротьби: аудит моделі на спеціальних наборах (StereoSet, CrowS-Pairs, WinoBias); очищення й збалансування навчальних даних; контрфактичне розширення даних (заміна he ↔ she); донавчання зі зменшенням упередженості (debiasing); фільтрація й модерація виходу; документування обмежень у model card; людський контроль у критичних рішеннях.

**5. Для чого в секції «Токенізація пари речень» виводяться числа 0 і 1?**

Це token_type_ids (segment ids). Вони вказують, до якого речення належить токен: 0 — [CLS], токени першого речення та перший [SEP]; 1 — токени другого речення та останній [SEP]. Оскільки обидва речення подаються як одна послідовність, саме за цими числами модель розрізняє їх. У виводі перші 20 значень — нулі (початок першого речення), останні 20 — одиниці (кінець другого).

**6. Що позначають числа 768 і 2, що виводяться після завантаження AutoModelForSequenceClassification.from_pretrained()?**

Це параметри нової голови: Linear(in_features=768, out_features=2). 768 — розмірність прихованого стану BERT-base (розмір вектора [CLS], який іде на вхід голови). 2 — кількість класів (num_labels=2): not_equivalent і equivalent. Голова містить 768·2 + 2 = 1538 параметрів, що збігається з виводом попередньої клітинки.

**7. Те що в кістяку набагато більше параметрів ніж в голові, це добре чи погано в нашому випадку?**

Загалом добре. У кістяку 109 482 240 параметрів, у голові — 1 538 (приблизно у 71 тисячу разів менше). Вся «мовна» компетентність уже міститься в кістяку після попереднього навчання, тому для нашої задачі голова має лише перетворити вектор [CLS] на два класи, і для цього вистачає невеликого датасету (3668 пар). Недолік: при повному донавчанні оновлюються всі 109 млн параметрів, тому потрібно більше пам'яті й часу, а на малих даних є ризик перенавчання та забування (див. п. 9). Саме тому використовують малий learning rate (2e-5).

**8. Що саме вимірюється за допомогою метрик accuracy & F1 в 01_bert_head?**

Accuracy — частка пар з валідаційної вибірки (408 прикладів), для яких передбачений клас збігається з міткою (0.860 — 86 % правильних відповідей). F1 — гармонійне середнє precision і recall для класу «equivalent» (мітка 1); отримано 0.902. F1 потрібна, бо класи в MRPC незбалансовані (приблизно дві третини пар — парафрази), і лише за accuracy можна переоцінити якість моделі. F1 показує, наскільки добре модель знаходить парафрази й не плутає з ними різні за змістом речення.

**9. Збільшіть кількість епох тренування, покажіть що відбувається і чому.**

Перенавчав нову модель із тими ж гіперпараметрами, але num_train_epochs=10 (клітинка вище). Результати на валідації:

| Епоха | Training loss | Validation loss | Accuracy | F1 |
|---|---|---|---|---|
| 1 | 0.4716 | 0.3986 | 0.7990 | 0.8464 |
| 2 | 0.3180 | 0.3823 | 0.8456 | 0.8930 |
| 3 | 0.1695 | 0.4927 | 0.8431 | 0.8824 |
| 4 | 0.1123 | 0.6096 | 0.8652 | 0.9057 |
| 5 | 0.0307 | 0.6658 | 0.8676 | 0.9036 |
| 6 | 0.0338 | 0.8651 | 0.8554 | 0.8985 |
| 7 | 0.0288 | 0.9545 | 0.8505 | 0.8968 |
| 8 | 0.0038 | 0.8783 | 0.8554 | 0.8977 |
| 9 | 0.0003 | 0.9435 | 0.8627 | 0.9041 |
| 10 | 0.0003 | 0.9310 | 0.8603 | 0.9022 |

Training loss падає майже до нуля (0.47 → 0.0003), а validation loss після 2-ї епохи зростає (0.38 → 0.93), тобто більш ніж удвічі. Це перенавчання (overfitting): модель з 109 млн параметрів просто запам'ятовує 3668 тренувальних прикладів, стає надто впевненою й помиляється на нових, причому дуже впевнено, що сильно збільшує loss. Accuracy й F1 при цьому не покращуються, а коливаються навколо 0.86 / 0.90, адже вони враховують лише вибраний клас, а не впевненість. Тренування при цьому тривало втричі довше (3 хв 14 с проти 1 хв 3 с) без користі. Висновок: для цієї задачі достатньо 2–3 епох (як у демо), а більше — тільки шкодить; на практиці варто використовувати early stopping за validation loss. Значення на 3-й епосі тут відрізняються від першого запуску, бо при 10 епохах інший графік затухання learning rate.

**10. Додайте ще 4 перевірки на власних даних в пункті 4.**

Додав клітинку з 4 новими парами (вона розташована в кінці п. 4). Результати:

- «The city council approved the new budget on Monday.» / «On Monday the city council passed the new budget.» → 99.0 % (парафраз, правильно);
- та сама перша фраза та «The football team won the championship last year.» → 3.1 % (різні за змістом, правильно);
- «He is happy with the results of the exam.» / «He is not happy with the results of the exam.» → 99.1 % (помилка: речення протилежні за змістом, але майже збігаються за словами, тому модель вважає їх парафразами);
- «Apple released a new iPhone in September.» / «A new iPhone was launched by Apple in September.» → 98.8 % (парафраз з переставленням слів та іншим дієсловом, правильно).

Висновок: модель добре розпізнає перефразування й явно різні теми, але помиляється на парах із запереченням: їй легко «обдурити» великим перетином слів.